# Formation Energy

Calculate the formation energy of a compound using a DFT workflow on the Mat3ra platform.

This notebook loads a compound material, resolves Standata elemental reference materials for its constituent elements, and checks that refined `total_energy` properties already exist for each elemental reference.

If any elemental reference material or elemental total energy is missing, the notebook stops and points to the Total Energy notebook. It does not create or run elemental pre-calculations automatically.

<h2 style="color:green">Usage</h2>

1. Put a compound material JSON into `../uploads`, or set `MATERIAL_NAME` to match a compound in Standata or on the platform.
2. Set the material and workflow parameters in cell 1.2 below.
3. Click "Run" > "Run All".
4. If elemental total energies are missing, run the Total Energy notebook for the corresponding Standata elemental material(s) first, then rerun this notebook.
5. Inspect the elemental references, their total energies used by the workflow, and the final formation energy.

## Summary

1. Set up the environment and parameters.
2. Authenticate and initialize API client.
3. Load compound material and resolve unique elements.
4. Resolve Standata elemental reference materials and check refined elemental total energies.
5. Configure and save the Formation Energy workflow.
6. Configure compute.
7. Create, submit, and monitor the Formation Energy job.
8. Retrieve results.


## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)


In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Set parameters and configurations for the workflow and job


In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
# Set organization name to use it as the owner, otherwise your personal account is used
ORGANIZATION_NAME = None

# 3. Material parameters
FOLDER = "../uploads"
MATERIAL_NAME = "Silicon"  # Name of the compound to load from uploads folder, Standata, or platform

# 4. Workflow parameters
WORKFLOW_SEARCH_TERM = "formation_energy.json"  # Search term for Workflows Standata
APPLICATION_NAME = "espresso"  # Application for the QE formation-energy subworkflow
SCF_KGRID = None  # e.g., [10, 10, 10]
MY_WORKFLOW_NAME = "Formation Energy"

# 5. Elemental total energy source
# Controls whose total_energy properties are used for elemental references:
#   'public'     — any owner (highest precision wins)
#   'curators'   — only curators' properties
#   'my_account' — curators' or your own properties
ELEMENTAL_TE_SOURCE = "public"

# 5b. Property group filter for method consistency
# Ensures elemental total energies come from the same computational method.
# Format: {application_short_name}:{model_type}:{model_subtype}
COMPOUND_GROUP = "qe:dft:gga:pbe"  # Quantum ESPRESSO + DFT/GGA/PBE

# 6. Compute parameters
CLUSTER_NAME = None  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.D
PPN = 1

# 7. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30  # seconds


## 2. Authenticate and initialize API client
### 2.1. Authenticate
Authenticate in the browser and have credentials stored in environment variable `OIDC_ACCESS_TOKEN`.


In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client


In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client


### 2.3. Select account to work under


In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")


### 2.4. Select project


In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")


## 3. Load compound material
### 3.1. Load material from local file, Standata, or platform


In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize
from mat3ra.notebooks_utils.material import load_material_from_folder

material = load_material_from_folder(FOLDER, MATERIAL_NAME) or Material.create(
    Materials.get_by_name_first_match(MATERIAL_NAME))

visualize(material)


### 3.2. Resolve unique elements in the compound


In [ ]:
elements = sorted(set(material.basis.elements.values))
print(f"Compound elements: {elements}")


## 4. Resolve elemental reference materials and check total energies
### 4.1. Resolve Standata elemental reference materials


In [ ]:
elemental_materials_data = client.materials.list(
    {"tags": "elemental", "metadata.element": {"$in": elements}},
)

element_materials_by_symbol = {
    el: [m for m in elemental_materials_data if m.get("metadata", {}).get("element") == el]
    for el in elements
}
element_material_ids_by_symbol = {
    el: [m["exabyteId"] for m in element_materials_by_symbol[el]]
    for el in elements
}
element_materials = {
    el: element_materials_by_symbol[el][0]
    for el in elements if element_materials_by_symbol[el]
}

missing = [el for el in elements if not element_material_ids_by_symbol.get(el)]
if missing:
    raise RuntimeError(f"Missing elemental reference material(s) for {missing}")

print(f"Resolved elemental materials for: {', '.join(elements)}")


### 4.2. Check refined elemental total energies


In [ ]:
def get_elemental_total_energy_holders(
    api_client, symbols, element_material_ids_by_symbol,
    owner_slug, compound_group, source="public"
):
    holders = {}
    for symbol in symbols:
        query = {
            "exabyteId": {"$in": element_material_ids_by_symbol[symbol]},
            "slug": "total_energy",
            "group": compound_group,
        }
        if source == "my_account":
            query["owner.slug"] = {"$in": [owner_slug, "curators"]}
        elif source == "curators":
            query["owner.slug"] = "curators"

        props = api_client.properties.list(
            query=query,
            projection={"sort": {"precision.value": -1}, "limit": 1},
        )
        holders[symbol] = props[0] if props else None
    return holders


elemental_total_energy_holders = get_elemental_total_energy_holders(
    client, elements, element_material_ids_by_symbol,
    owner_slug=selected_account.entity_cache.get("slug", selected_account.name),
    compound_group=COMPOUND_GROUP,
    source=ELEMENTAL_TE_SOURCE,
)

missing = [s for s in elements if elemental_total_energy_holders[s] is None]
if missing:
    raise RuntimeError(f"Missing total_energy for {missing}. Run total_energy.ipynb first.")

for symbol in elements:
    h = elemental_total_energy_holders[symbol]
    print(
        f"♻️  {symbol}: {h['data']['value']} eV "
        f"(precision: {h['precision']['value']}, group: {COMPOUND_GROUP})"
    )


### 4.3. Save compound material for the workflow


In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_material_response = get_or_create_material(client, material, ACCOUNT_ID)
saved_material = Material.create(saved_material_response)

print(f"✅ Saved compound material: {saved_material_response['_id']}")

## 5. Configure the Formation Energy workflow
### 5.1. Select application


In [ ]:
from mat3ra.ade.application import Application
from mat3ra.standata.applications import ApplicationStandata

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
print(f"Using application: {app.name}")


### 5.2. Load workflow from Standata and preview it


In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.context.providers import PointsGridDataProvider
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

formation_workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(
    WORKFLOW_SEARCH_TERM
)
formation_workflow = Workflow.create(formation_workflow_config)
formation_workflow.name = MY_WORKFLOW_NAME

if SCF_KGRID is not None:
    new_context = PointsGridDataProvider(dimensions=SCF_KGRID, isEdited=True).get_context_item_data()
    for subworkflow in formation_workflow.subworkflows:
        unit = subworkflow.get_unit_by_name(name="pw_scf")
        if unit:
            unit.add_context(new_context)
            subworkflow.set_unit(unit)

visualize_workflow(formation_workflow)


### 5.3. Save workflow to collection


In [ ]:
from mat3ra.notebooks_utils.core.entity.workflow.api import get_or_create_workflow

saved_formation_workflow_response = get_or_create_workflow(
    client, formation_workflow, ACCOUNT_ID
)
saved_formation_workflow = Workflow.create(saved_formation_workflow_response)
print(f"Formation workflow ID: {saved_formation_workflow.id}")


## 6. Create the compute configuration
### 6.1. Select cluster


In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")


### 6.2. Create compute configuration


In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")


## 7. Create the Formation Energy job
### 7.1. Create job with compound material and workflow configuration


In [ ]:
from mat3ra.notebooks_utils.job import create_job

formation_job_name = f"{MY_WORKFLOW_NAME} {saved_material.formula} {timestamp}"
formation_job_response = create_job(
    api_client=client,
    materials=[saved_material],
    workflow=formation_workflow,
    project_id=project_id,
    owner_id=ACCOUNT_ID,
    prefix=formation_job_name,
    compute=compute.to_dict(),
)

formation_job_id = formation_job_response["_id"]
print(f"✅ Formation Energy job created: {formation_job_id}")


### 7.2. Submit the Formation Energy job and monitor the status


In [ ]:
client.jobs.submit(formation_job_id)
print(f"✅ Job {formation_job_id} submitted successfully!")


In [ ]:
from mat3ra.notebooks_utils.api.job import wait_for_jobs_to_finish_async

await wait_for_jobs_to_finish_async(client.jobs, formation_job_id, poll_interval=POLL_INTERVAL)


## 8. Retrieve results
### 8.1. Retrieve and visualize Formation Energy results


In [ ]:
from mat3ra.notebooks_utils.ipython.entity.property.visualize import visualize_properties

formation_energy_data = client.properties.get_for_job(formation_job_id)
visualize_properties(formation_energy_data)
